# ADWIN: Adaptive Windowing for Streaming Drift Detection

## Learning Objectives

In this notebook, you will learn:
- How the ADWIN algorithm works for online drift detection
- How to implement ADWIN using the River library
- How to tune ADWIN parameters (delta, clock, min_window_length)
- How to apply ADWIN to real-time streaming data
- How ADWIN compares to other online drift detection methods

## Introduction

**ADWIN (ADaptive WINdowing)** is a popular online drift detection method with mathematical guarantees. It is designed for streaming data where observations arrive one at a time and drift needs to be detected in real-time.

### How ADWIN Works

1. **Maintains a variable-length window** of recent data points
2. **Divides the window into two sub-windows** (W0 and W1)
3. **Compares the means** of the two sub-windows
4. **Detects drift** when the means are significantly different
5. **Shrinks the window** by removing old data when drift is detected

### Key Parameters

- **delta**: Significance level (controls false positive rate)
- **clock**: How often to check for drift (1 = every point, higher = less frequent)
- **min_window_length**: Minimum window size for drift detection
- **grace_period**: Number of points before drift detection starts

In [ ]:
# Install River library if not already installed
# !pip install river

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from river import drift

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Generate Streaming Data with Drift

We'll create a synthetic data stream with different types of drift patterns.

In [ ]:
def generate_stream_with_drift(n_samples=2000, drift_type='abrupt', drift_position=1000):
    """
    Generate a data stream with drift.
    
    Parameters:
    -----------
    n_samples : int
        Total number of samples
    drift_type : str
        Type of drift: 'abrupt', 'gradual', 'incremental', or 'recurring'
    drift_position : int
        Position where drift starts
    
    Returns:
    --------
    stream : array
        Data stream with drift
    """
    stream = np.zeros(n_samples)
    
    if drift_type == 'abrupt':
        # Abrupt drift: sudden change
        stream[:drift_position] = np.random.normal(0, 1, drift_position)
        stream[drift_position:] = np.random.normal(2, 1, n_samples - drift_position)
    
    elif drift_type == 'gradual':
        # Gradual drift: slow transition
        transition_length = 500
        stream[:drift_position] = np.random.normal(0, 1, drift_position)
        
        # Gradual transition
        for i in range(drift_position, min(drift_position + transition_length, n_samples)):
            progress = (i - drift_position) / transition_length
            mean = 0 + progress * 2  # Gradually shift from 0 to 2
            stream[i] = np.random.normal(mean, 1)
        
        # After transition
        if drift_position + transition_length < n_samples:
            stream[drift_position + transition_length:] = np.random.normal(2, 1, 
                                                                           n_samples - drift_position - transition_length)
    
    elif drift_type == 'incremental':
        # Incremental drift: series of small steps
        step_size = 100
        n_steps = 5
        current_mean = 0
        
        for step in range(n_steps + 1):
            start_idx = drift_position + step * step_size
            end_idx = min(drift_position + (step + 1) * step_size, n_samples)
            
            if start_idx < n_samples:
                stream[start_idx:end_idx] = np.random.normal(current_mean, 1, end_idx - start_idx)
                current_mean += 0.4  # Increment mean by 0.4 each step
        
        # Before drift
        stream[:drift_position] = np.random.normal(0, 1, drift_position)
    
    elif drift_type == 'recurring':
        # Recurring drift: periodic pattern
        period = 400
        for i in range(n_samples):
            cycle_position = i % period
            if cycle_position < period // 2:
                stream[i] = np.random.normal(0, 1)
            else:
                stream[i] = np.random.normal(2, 1)
    
    return stream

# Generate different drift scenarios
stream_abrupt = generate_stream_with_drift(drift_type='abrupt')
stream_gradual = generate_stream_with_drift(drift_type='gradual')
stream_incremental = generate_stream_with_drift(drift_type='incremental')
stream_recurring = generate_stream_with_drift(drift_type='recurring')

print("Streaming data generated successfully!")
print(f"Stream length: {len(stream_abrupt)} samples")

## 2. Visualize Drift Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Abrupt drift
axes[0, 0].plot(stream_abrupt, alpha=0.7, linewidth=0.5)
axes[0, 0].axvline(x=1000, color='red', linestyle='--', label='Drift Point')
axes[0, 0].set_title('Abrupt Drift', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Value')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Gradual drift
axes[0, 1].plot(stream_gradual, alpha=0.7, linewidth=0.5, color='orange')
axes[0, 1].axvline(x=1000, color='red', linestyle='--', label='Drift Start')
axes[0, 1].axvline(x=1500, color='green', linestyle='--', label='Drift End')
axes[0, 1].set_title('Gradual Drift', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Time')
axes[0, 1].set_ylabel('Value')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Incremental drift
axes[1, 0].plot(stream_incremental, alpha=0.7, linewidth=0.5, color='green')
axes[1, 0].axvline(x=1000, color='red', linestyle='--', label='Drift Start')
axes[1, 0].set_title('Incremental Drift', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_ylabel('Value')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recurring drift
axes[1, 1].plot(stream_recurring, alpha=0.7, linewidth=0.5, color='purple')
axes[1, 1].set_title('Recurring/Seasonal Drift', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Value')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Implement ADWIN Drift Detection

In [ ]:
def detect_drift_adwin(stream, delta=0.002, clock=32, min_window_length=5, grace_period=10):
    """
    Detect drift in a data stream using ADWIN.
    
    Parameters:
    -----------
    stream : array-like
        Data stream
    delta : float
        Significance level (smaller = more sensitive)
    clock : int
        How often to check for drift
    min_window_length : int
        Minimum window size
    grace_period : int
        Number of points before drift detection starts
    
    Returns:
    --------
    drift_points : list
        Indices where drift was detected
    window_sizes : list
        Window size at each time step
    estimations : list
        Mean estimation at each time step
    """
    adwin = drift.ADWIN(delta=delta, clock=clock, min_window_length=min_window_length, 
                        grace_period=grace_period)
    
    drift_points = []
    window_sizes = []
    estimations = []
    
    for i, value in enumerate(stream):
        adwin.update(value)
        
        if adwin.drift_detected:
            drift_points.append(i)
        
        window_sizes.append(adwin.width)
        estimations.append(adwin.estimation)
    
    return drift_points, window_sizes, estimations

# Apply ADWIN to abrupt drift
drift_points_abrupt, window_sizes_abrupt, estimations_abrupt = detect_drift_adwin(stream_abrupt)

print(f"Drift detected at positions: {drift_points_abrupt}")
print(f"Total drift detections: {len(drift_points_abrupt)}")

## 4. Visualize ADWIN Detection Results

In [ ]:
def visualize_adwin_results(stream, drift_points, window_sizes, estimations, title):
    """
    Visualize ADWIN drift detection results.
    """
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    
    # Plot 1: Data stream with drift points
    axes[0].plot(stream, alpha=0.7, linewidth=0.5, label='Data Stream')
    for dp in drift_points:
        axes[0].axvline(x=dp, color='red', linestyle='--', alpha=0.7)
    axes[0].set_title(f'{title} - Data Stream with Detected Drift Points', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Time')
    axes[0].set_ylabel('Value')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Window size evolution
    axes[1].plot(window_sizes, color='green', linewidth=1.5, label='Window Size')
    for dp in drift_points:
        axes[1].axvline(x=dp, color='red', linestyle='--', alpha=0.7)
    axes[1].set_title('ADWIN Window Size Evolution', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Window Size')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Mean estimation
    axes[2].plot(estimations, color='purple', linewidth=1.5, label='Mean Estimation')
    axes[2].plot(stream, alpha=0.3, linewidth=0.5, label='Actual Data', color='gray')
    for dp in drift_points:
        axes[2].axvline(x=dp, color='red', linestyle='--', alpha=0.7, label='Drift' if dp == drift_points[0] else '')
    axes[2].set_title('ADWIN Mean Estimation', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Time')
    axes[2].set_ylabel('Mean')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualize abrupt drift detection
visualize_adwin_results(stream_abrupt, drift_points_abrupt, window_sizes_abrupt, 
                       estimations_abrupt, 'Abrupt Drift')

## 5. Test ADWIN on Different Drift Patterns

In [ ]:
# Gradual drift
drift_points_gradual, window_sizes_gradual, estimations_gradual = detect_drift_adwin(stream_gradual)
print(f"Gradual Drift - Detected at: {drift_points_gradual}")
visualize_adwin_results(stream_gradual, drift_points_gradual, window_sizes_gradual, 
                       estimations_gradual, 'Gradual Drift')

In [ ]:
# Incremental drift
drift_points_incremental, window_sizes_incremental, estimations_incremental = detect_drift_adwin(stream_incremental)
print(f"Incremental Drift - Detected at: {drift_points_incremental}")
visualize_adwin_results(stream_incremental, drift_points_incremental, window_sizes_incremental, 
                       estimations_incremental, 'Incremental Drift')

In [ ]:
# Recurring drift
drift_points_recurring, window_sizes_recurring, estimations_recurring = detect_drift_adwin(stream_recurring)
print(f"Recurring Drift - Detected at: {drift_points_recurring[:10]}... (showing first 10)")
print(f"Total detections: {len(drift_points_recurring)}")
visualize_adwin_results(stream_recurring, drift_points_recurring, window_sizes_recurring, 
                       estimations_recurring, 'Recurring Drift')

## 6. Parameter Sensitivity Analysis

In [ ]:
# Test different delta values
delta_values = [0.0001, 0.001, 0.002, 0.01, 0.05, 0.1]
detection_counts = []
detection_delays = []

true_drift_position = 1000

for delta in delta_values:
    drift_points, _, _ = detect_drift_adwin(stream_abrupt, delta=delta)
    detection_counts.append(len(drift_points))
    
    # Calculate detection delay (first detection after true drift)
    detections_after_drift = [dp for dp in drift_points if dp >= true_drift_position]
    if detections_after_drift:
        delay = detections_after_drift[0] - true_drift_position
    else:
        delay = None
    detection_delays.append(delay)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Number of detections vs delta
axes[0].plot(delta_values, detection_counts, marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('Delta (Significance Level)')
axes[0].set_ylabel('Number of Drift Detections')
axes[0].set_title('Sensitivity: Number of Detections vs Delta')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# Detection delay vs delta
valid_delays = [(d, delay) for d, delay in zip(delta_values, detection_delays) if delay is not None]
if valid_delays:
    deltas_with_delay, delays = zip(*valid_delays)
    axes[1].plot(deltas_with_delay, delays, marker='o', linewidth=2, markersize=8, color='orange')
    axes[1].set_xlabel('Delta (Significance Level)')
    axes[1].set_ylabel('Detection Delay (samples)')
    axes[1].set_title('Detection Delay vs Delta')
    axes[1].set_xscale('log')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nParameter Sensitivity Results:")
print("=" * 60)
for delta, count, delay in zip(delta_values, detection_counts, detection_delays):
    print(f"Delta: {delta:.4f} | Detections: {count} | Delay: {delay if delay is not None else 'N/A'}")

## 7. Real-World Example: IoT Sensor Monitoring

In [ ]:
# Simulate IoT temperature sensor data
# Normal operation: temperature around 25°C
# Anomaly: sudden temperature spike (equipment malfunction)

n_samples = 3000
sensor_data = np.zeros(n_samples)

# Normal operation
sensor_data[:1500] = np.random.normal(25, 1, 1500)

# Equipment malfunction (gradual temperature increase)
for i in range(1500, 2000):
    progress = (i - 1500) / 500
    mean = 25 + progress * 10  # Temperature rises from 25 to 35
    sensor_data[i] = np.random.normal(mean, 1.5)

# Critical failure (high temperature)
sensor_data[2000:] = np.random.normal(35, 2, 1000)

# Apply ADWIN
drift_points_sensor, window_sizes_sensor, estimations_sensor = detect_drift_adwin(
    sensor_data, delta=0.002, clock=10
)

print("IoT Sensor Monitoring Results")
print("=" * 60)
print(f"Drift detected at time steps: {drift_points_sensor}")
print(f"\nInterpretation:")
if drift_points_sensor:
    print(f"  - First drift detected at sample {drift_points_sensor[0]}")
    print(f"  - This corresponds to {drift_points_sensor[0] / 60:.1f} minutes into monitoring")
    print(f"  - Alert: Equipment malfunction detected!")
    print(f"  - Recommendation: Immediate inspection required")

# Visualize
visualize_adwin_results(sensor_data, drift_points_sensor, window_sizes_sensor, 
                       estimations_sensor, 'IoT Temperature Sensor Monitoring')

## 8. Comparison with Other Online Drift Detectors

In [ ]:
# Compare ADWIN with Page-Hinkley test
from river.drift import PageHinkley

def detect_drift_page_hinkley(stream, min_instances=30, delta=0.005, threshold=50):
    """
    Detect drift using Page-Hinkley test.
    """
    ph = PageHinkley(min_instances=min_instances, delta=delta, threshold=threshold)
    drift_points = []
    
    for i, value in enumerate(stream):
        ph.update(value)
        if ph.drift_detected:
            drift_points.append(i)
    
    return drift_points

# Apply both methods to abrupt drift
drift_adwin = drift_points_abrupt
drift_ph = detect_drift_page_hinkley(stream_abrupt)

print("Comparison: ADWIN vs Page-Hinkley")
print("=" * 60)
print(f"ADWIN detections: {len(drift_adwin)} at {drift_adwin}")
print(f"Page-Hinkley detections: {len(drift_ph)} at {drift_ph}")

# Visualize comparison
plt.figure(figsize=(16, 6))
plt.plot(stream_abrupt, alpha=0.7, linewidth=0.5, label='Data Stream')
plt.axvline(x=1000, color='black', linestyle=':', linewidth=2, label='True Drift')

for dp in drift_adwin:
    plt.axvline(x=dp, color='red', linestyle='--', alpha=0.7, 
               label='ADWIN' if dp == drift_adwin[0] else '')

for dp in drift_ph:
    plt.axvline(x=dp, color='blue', linestyle='-.', alpha=0.7, 
               label='Page-Hinkley' if dp == drift_ph[0] else '')

plt.title('Drift Detection Comparison: ADWIN vs Page-Hinkley', fontsize=14, fontweight='bold')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Key Takeaways

### ADWIN Strengths

1. **Mathematical guarantees**: Provides theoretical bounds on false positive and false negative rates
2. **Adaptive window**: Automatically adjusts window size based on drift
3. **No assumptions**: Works with any data distribution
4. **Real-time**: Processes data points one at a time
5. **Multiple drift types**: Can detect abrupt, gradual, and incremental drift

### Parameter Tuning Guidelines

| Parameter | Effect | Recommendation |
|-----------|--------|----------------|
| **delta** | Smaller = more sensitive | 0.002 (default) for balanced sensitivity |
| **clock** | Higher = less frequent checks | 32 (default) for efficiency, 1 for maximum responsiveness |
| **min_window_length** | Larger = more stable | 5-10 for most applications |
| **grace_period** | Larger = more initial data | 10-50 depending on data characteristics |

### When to Use ADWIN

- **Streaming data**: When data arrives continuously
- **Real-time detection**: When immediate drift detection is critical
- **Unknown drift patterns**: When drift type is unpredictable
- **Resource constraints**: When computational efficiency matters
- **Mathematical rigor**: When theoretical guarantees are required

### Practical Considerations

1. **False positives**: Lower delta increases sensitivity but may cause false alarms
2. **Detection delay**: Trade-off between sensitivity and delay
3. **Computational cost**: Clock parameter controls overhead
4. **Window size**: Monitor window size evolution for insights
5. **Combine with other methods**: Use multiple detectors for robustness

## Exercises

1. Implement a custom drift detector that combines ADWIN with a performance metric (e.g., accuracy).

2. Create a simulation where you vary the drift magnitude and measure ADWIN's detection delay.

3. Build a real-time dashboard that visualizes ADWIN's window size and mean estimation.

4. Compare ADWIN with KSWIN (Kolmogorov-Smirnov Windowing) on the same datasets.